In [18]:
# =============================================================================
# SHOW REEL MEDIA GROUP — COMMUNITY PERSONA IDENTIFICATION PIPELINE
# Platform: Instagram Only
# Runtime: Google Colab Enterprise + Vertex AI (Gemini)
# Author: Show Reel Data Science Team
# Version: 1.0
#
# Pipeline Stages:
#   Stage 0 — Feature Engineering (user-level aggregation from ig_comments + ig_media)
#   Stage 1 — Exploratory Taxonomy Discovery (Gemini Flash, sample ~500 users)
#   Stage 2 — Deterministic CAG Classification (Gemini Pro, full dataset)
#
# Scheduling: Designed for Colab Enterprise Scheduled Runtime.
#   Set PIPELINE_MODE = "SAMPLE"  → Stage 0 + Stage 1 only (taxonomy discovery)
#   Set PIPELINE_MODE = "FULL"    → Stage 0 + Stage 2 only (production classification)
#   Set PIPELINE_MODE = "ALL"     → All stages in sequence
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────────────────────
# Run this cell once per runtime, then restart.
!pip install -q google-cloud-aiplatform pandas numpy tqdm langdetect emoji

import subprocess, sys
# Uncomment below if running in a fresh Colab environment:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "google-cloud-aiplatform", "pandas", "numpy", "tqdm",
                 "langdetect", "emoji"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'google-cloud-aiplatform', 'pandas', 'numpy', 'tqdm', 'langdetect', 'emoji'], returncode=0)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — CONFIGURATION (Toggle models and modes here)
# ─────────────────────────────────────────────────────────────────────────────

import os

# ── GCP / Vertex AI ──────────────────────────────────────────────────────────
GCP_PROJECT_ID   = "gen-lang-client-0792749758"          # TODO: set your project
GCP_LOCATION     = "us-central1"                   # Vertex AI region
GCS_BUCKET       = "afb_showreel"     # GCS path for outputs

# ── Model Toggle ─────────────────────────────────────────────────────────────
# Stage 1 (Exploratory): Flash is cheaper and fast for discovery on a sample
# Stage 2 (Deterministic): Pro for higher reasoning quality on full dataset
MODEL_STAGE1_EXPLORATORY = "gemini-2.5-flash"
MODEL_STAGE2_CLASSIFY    = "gemini-2.5-pro"
# Swap the above to "gemini-2.0-flash-001" for both if cost is a constraint.

# ── Determinism Config (mandatory for academic reproducibility) ───────────────
TEMPERATURE = 0.0   # Greedy decoding — zero stochasticity
TOP_P       = 1.0   # No nucleus truncation; greedy is sole selector
MAX_OUTPUT_TOKENS_STAGE1 = 2048
MAX_OUTPUT_TOKENS_STAGE2 = 512   # Classification outputs are compact

# ── Pipeline Mode ────────────────────────────────────────────────────────────
# "SAMPLE" → Stage 0 + Stage 1 (taxonomy discovery, human-in-the-loop)
# "FULL"   → Stage 0 + Stage 2 (production classification, requires taxonomy JSON)
# "ALL"    → Both stages in sequence (not recommended for large datasets)
PIPELINE_MODE = "SAMPLE"   # Toggle here

# ── Data Paths ────────────────────────────────────────────────────────────────
# Single unified LLM JSONL file from data preparation pipeline (contains platform field)
COMMENTS_LLM_PATH = f"gs://{GCS_BUCKET}/comment_prep/2131052821112422400/final_output/comments_llm.jsonl"
# Toggle which platforms to include (data will be filtered by platform field)
PLATFORMS_TO_INCLUDE = ["instagram", "tiktok"]

# ── Sampling Config ───────────────────────────────────────────────────────────
SAMPLE_N_USERS        = 500    # Stage 1: number of users for taxonomy discovery
SAMPLE_SEED           = 42
BATCH_SIZE_STAGE1     = 20     # Users per exploratory LLM call (taxonomy)
BATCH_SIZE_STAGE2     = 10     # Users per classification call

# ── Artifact Paths ────────────────────────────────────────────────────────────
TAXONOMY_JSON_PATH    = "outputs/taxonomy.json"
CAG_CACHE_DIR         = "outputs/cag_cache/"       # Pickled CAG objects per post
RESULTS_PATH          = "outputs/user_personas.parquet"
os.makedirs("outputs", exist_ok=True)
os.makedirs(CAG_CACHE_DIR, exist_ok=True)

print("✅ Configuration loaded.")
print(f"   Mode: {PIPELINE_MODE}")
print(f"   Stage 1 model: {MODEL_STAGE1_EXPLORATORY}")
print(f"   Stage 2 model: {MODEL_STAGE2_CLASSIFY}")
print(f"   Platforms: {', '.join(PLATFORMS_TO_INCLUDE)}")
print(f"   Comments source: {COMMENTS_LLM_PATH}")


In [20]:
# prompt: test whether the llms of the two stages would work on vertex ai

from google.cloud import aiplatform
from vertexai.generative_models import GenerativeModel, Part

# Initialize Vertex AI
aiplatform.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

# Load models
model_stage1 = GenerativeModel(MODEL_STAGE1_EXPLORATORY)
model_stage2 = GenerativeModel(MODEL_STAGE2_CLASSIFY)

# Define a simple prompt for testing
test_prompt = "What is the capital of France?"

print("Testing Stage 1 Model...")
try:
    response_stage1 = model_stage1.generate_content(
        test_prompt,
        generation_config={
            "max_output_tokens": MAX_OUTPUT_TOKENS_STAGE1,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
        }
    )
    print(f"Stage 1 Model Response: {response_stage1.text}")
    print("✅ Stage 1 model is working.")
except Exception as e:
    print(f"❌ Error testing Stage 1 model: {e}")

print("\nTesting Stage 2 Model...")
try:
    response_stage2 = model_stage2.generate_content(
        test_prompt,
        generation_config={
            "max_output_tokens": MAX_OUTPUT_TOKENS_STAGE2,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
        }
    )
    print(f"Stage 2 Model Response: {response_stage2.text}")
    print("✅ Stage 2 model is working.")
except Exception as e:
    print(f"❌ Error testing Stage 2 model: {e}")


Testing Stage 1 Model...


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Stage 1 Model Response: The capital of France is **Paris**.
✅ Stage 1 model is working.

Testing Stage 2 Model...
Stage 2 Model Response: The capital of France is **Paris**.
✅ Stage 2 model is working.


In [21]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — VERTEX AI INITIALIZATION
# ─────────────────────────────────────────────────────────────────────────────

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig, Part

vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

def get_model(model_name: str) -> GenerativeModel:
    """Returns a configured GenerativeModel for the given model name."""
    return GenerativeModel(model_name)

def get_generation_config(max_tokens: int) -> GenerationConfig:
    """Returns a deterministic GenerationConfig."""
    return GenerationConfig(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_output_tokens=max_tokens,
        response_mime_type="application/json",  # Enforce JSON output at API level
    )

print("✅ Vertex AI initialized.")

✅ Vertex AI initialized.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — DATA LOADING & PLATFORM-BASED PARSING
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import os
import json
from typing import List

def load_and_parse_comments_by_platform(
    gcs_path: str, 
    platforms_to_include: List[str]
) -> pd.DataFrame:
    """
    Load unified LLM JSONL from GCS, filter by platform field.
    """
    print(f"\n📥 Loading comments from: {gcs_path}")
    print(f"   Filtering for platforms: {', '.join(platforms_to_include)}")
    
    try:
        # Load entire JSONL file
        df = pd.read_json(gcs_path, lines=True)
        print(f"   Total rows in file: {len(df):,}")
        
        # Check for platform field
        if "platform" not in df.columns:
            raise ValueError("❌ 'platform' field not found in comments_llm.jsonl")
        
        print(f"\n   Platforms in file: {df['platform'].unique().tolist()}")
        
        # Filter to requested platforms
        df = df[df["platform"].isin(platforms_to_include)].copy()
        print(f"   Filtered to {len(df):,} rows from {platforms_to_include}")
        
        # Validate required columns exist
        required_cols = ["comment_id", "text", "author_id", "media_id", "platform", "timestamp"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print(f"   ⚠️  Missing columns: {missing}")
            print(f"   Available columns: {df.columns.tolist()}")
            # Try to infer/rename if possible
            rename_attempts = {
                "text_original": "text",
                "comment_text": "text",
                "user_id": "author_id",
                "author_name": "author_id",
                "media_id": "media_id",
                "post_id": "media_id",
            }
            df.rename(columns={k: v for k, v in rename_attempts.items() if k in df.columns}, inplace=True)
        
        # Filter out null/empty text
        df = df[df["text"].notna() & (df["text"].astype(str).str.strip() != "")].copy()
        print(f"\n✅ Comments loaded: {len(df):,} rows")
        
        # Platform breakdown
        print(f"   Breakdown by platform:")
        for platform in sorted(df["platform"].unique()):
            count = (df["platform"] == platform).sum()
            pct = (count / len(df) * 100)
            print(f"      {platform.upper():<12}: {count:>8,} ({pct:>5.1f}%)")
        
        return df
    
    except Exception as e:
        print(f"❌ Error loading comments: {e}")
        raise


# ── EXECUTE: Load and parse comments by platform ──────────────────────────────
comments_df = load_and_parse_comments_by_platform(COMMENTS_LLM_PATH, PLATFORMS_TO_INCLUDE)

print(f"\n📋 Sample row from first comment:")
sample_row = comments_df.iloc[0].to_dict()
for key, val in list(sample_row.items())[:8]:
    print(f"   {key}: {str(val)[:80]}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3.5 — EARLY STAGE 0: COMMENT PROCESSING & JSONL EXPORT (MINIMAL)
# ─────────────────────────────────────────────────────────────────────────────
# Process comments preserving all original information:
# - Keep emoji (critical for persona detection)
# - Keep URLs (reveal engagement intent)
# - Minimal whitespace normalization only
# Export to JSONL for LLM + ML training

import json
import re
from typing import Optional, Dict, Any, List
import emoji as emoji_lib
import langdetect

def get_emoji_categories(text: str) -> List[str]:
    """Detect emoji categories in text (love, celebration, question, etc)."""
    categories = set()
    for em in emoji_lib.emoji_list(text):
        emoji_char = em['emoji']
        try:
            emoji_data = emoji_lib.emoji_dict[emoji_char]
            name = emoji_data.get('en', '').lower()

            if any(x in name for x in ['heart', 'love', 'kiss', 'cupid']):
                categories.add('love')
            elif any(x in name for x in ['laugh', 'joy', 'smile', 'grin', 'happy']):
                categories.add('celebration')
            elif any(x in name for x in ['question', 'thinking', 'confused']):
                categories.add('question')
            elif any(x in name for x in ['fire', 'explosion', 'star', 'sparkle']):
                categories.add('emphasis')
            elif any(x in name for x in ['cry', 'sad', 'tear', 'angry']):
                categories.add('negative')
            elif any(x in name for x in ['clap', 'pray', 'wave', 'thumbs', 'ok']):
                categories.add('gesture')
            else:
                categories.add('other')
        except:
            categories.add('other')

    return sorted(list(categories))


def minimal_text_preprocessing(text: str) -> Dict[str, Any]:
    """
    Minimal preprocessing: normalize whitespace only.
    Preserve: emoji, URLs, mentions, special characters.
    """
    if not isinstance(text, str) or not text.strip():
        return {
            "text": "",
            "text_clean_whitespace": "",
            "emoji_count": 0,
            "emoji_categories": [],
            "language": None,
            "has_links": False,
            "is_reply": False,
        }

    text_orig = text.strip()
    text_normalized = re.sub(r'\s+', ' ', text_orig).strip()

    emoji_count = emoji_lib.emoji_count(text_orig)
    emoji_categories = get_emoji_categories(text_orig) if emoji_count > 0 else []
    has_links = bool(re.search(r'https?://\S+|www\.\S+', text_orig))
    is_reply = bool(re.match(r'^\s*@\w+', text_orig))

    try:
        detected_lang = langdetect.detect(text_orig)
    except:
        detected_lang = None

    return {
        "text": text_orig,
        "text_clean_whitespace": text_normalized,
        "emoji_count": emoji_count,
        "emoji_categories": emoji_categories,
        "language": detected_lang,
        "has_links": has_links,
        "is_reply": is_reply,
    }


def export_comments_to_jsonl(
    comments_df,
    output_gcs_path: str,
    max_records: Optional[int] = None,
) -> int:
    """Export minimally processed comments to JSONL."""
    from google.cloud import storage

    print(f"\n📝 Processing {len(comments_df):,} comments for JSONL export...")

    df = comments_df.copy()
    if max_records:
        df = df.sample(n=min(max_records, len(df)), random_state=42)

    print("   Running minimal text preprocessing...")
    text_features = df["text"].apply(minimal_text_preprocessing)

    records = []
    for idx, (_, row) in enumerate(df.iterrows()):
        text_proc = text_features.iloc[idx]

        metadata = {
            "emoji_count": int(text_proc["emoji_count"]),
            "emoji_categories": text_proc["emoji_categories"],
            "mention_count": len(re.findall(r'@\w+', text_proc["text"])),
            "hashtag_count": len(re.findall(r'#\w+', text_proc["text"])),
            "url_count": len(re.findall(r'https?://\S+|www\.\S+', text_proc["text"])),
            "has_links": text_proc["has_links"],
            "language": text_proc["language"],
            "is_reply": text_proc["is_reply"],
            "timestamp": str(row.get("timestamp", "")),
            "likes": int(row.get("like_count", 0)),
            "media_id": str(row.get("media_id", "")),
        }

        record = {
            "id": str(row.get("comment_id", f"comment_{idx}")),
            "user_id": str(row.get("from_id", "")),
            "username": str(row.get("from_username", "")),
            "text": text_proc["text"],
            "text_clean_whitespace": text_proc["text_clean_whitespace"],
            "metadata": metadata,
        }
        records.append(record)

    jsonl_buffer = "\n".join(json.dumps(r, ensure_ascii=False) for r in records)

    print(f"   Uploading to {output_gcs_path}...")
    try:
        client = storage.Client()
        bucket_name = output_gcs_path.replace("gs://", "").split("/")[0]
        blob_path = "/".join(output_gcs_path.replace("gs://", "").split("/")[1:])

        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_path)
        blob.upload_from_string(jsonl_buffer, content_type="application/x-ndjson")

        print(f"   Exported {len(records):,} records to: {output_gcs_path}")
        return len(records)
    except Exception as e:
        print(f"   GCS upload failed: {e}")
        local_path = output_gcs_path.replace("gs://", "").replace("/", "_")
        with open(local_path, "w", encoding="utf-8") as f:
            f.write(jsonl_buffer)
        print(f"   Exported {len(records):,} records to: {local_path}")
        return len(records)


# ── EXECUTE: Export processed comments to JSONL ────────────────────────────────
print("Exporting minimally-processed comments to JSONL...")
export_comments_to_jsonl(
    comments_df=ig_comments,
    output_gcs_path=f"gs://{GCS_BUCKET}/ig_comments_for_llm.jsonl",
    max_records=None
)

print("\n✓ JSONL export complete. All original data preserved (emoji, URLs, etc).")


In [38]:
!gsutil ls gs://{GCS_BUCKET}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://afb_showreel/YTcomments_1_cleaned.parquet
gs://afb_showreel/YTcomments_2_cleaned.parquet
gs://afb_showreel/YTcomments_3_cleaned.parquet
gs://afb_showreel/YTcomments_4_cleaned.parquet
gs://afb_showreel/channels_metadata.csv
gs://afb_showreel/fb_comments_clean.parquet
gs://afb_showreel/fb_posts_clean.parquet
gs://afb_showreel/ig_comments_cleaned.parquet
gs://afb_showreel/ig_posts_cleaned.parquet
gs://afb_showreel/processed_urls.txt
gs://afb_showreel/tk_comments_clean.parquet
gs://afb_showreel/tk_posts_clean.parquet
gs://afb_showreel/undownloaded_links.csv
gs://afb_showreel/videos_metadata.csv
gs://afb_showreel/2519586145507999744/
gs://afb_showreel/3488985965299499008/
gs://afb_showreel/Output/
gs://afb_showreel/cluster_summarie

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — STAGE 0: USER AGGREGATION (FEATURES PRE-COMPUTED IN DATA PREP)
# ─────────────────────────────────────────────────────────────────────────────
# Data preparation pipeline already computed comment-level features (emoji, mention, etc).
# This stage only aggregates to user level: mean/sum across comments per user per platform.

import re

# ── 4.1 Load ML features from data preparation pipeline ──────────────────────
def load_ml_features_by_platform(
    gcs_bucket: str,
    platforms: list,
    base_path: str = "comment_prep/2131052821112422400/final_output"
) -> pd.DataFrame:
    """Load pre-computed ML features from data prep pipeline."""
    print(f"\n📥 Loading ML features (pre-computed by data prep pipeline)...")
    
    all_ml = []
    for platform in platforms:
        gcs_path = f"gs://{gcs_bucket}/{base_path}/comments_ml_{platform}.parquet"
        try:
            df = pd.read_parquet(gcs_path)
            df["platform"] = platform  # Ensure platform field
            all_ml.append(df)
            print(f"   {platform.upper()}: {len(df):,} rows loaded ✅")
        except Exception as e:
            print(f"   {platform.upper()}: Load failed — {e}")
    
    if not all_ml:
        raise ValueError("No ML features loaded from any platform")
    
    combined = pd.concat(all_ml, ignore_index=True)
    print(f"\n✅ Total: {len(combined):,} comment-level features")
    return combined


# Load ML features (not JSONL) for user aggregation
comment_features = load_ml_features_by_platform(GCS_BUCKET, PLATFORMS_TO_INCLUDE)

print(f"   Available features: {comment_features.columns.tolist()[:10]}...")


# ── 4.2 Merge with comment text for top comments sampling ────────────────────
# Load JSONL for text (needed for representative comments only)
print(f"\n📥 Loading comment text from LLM JSONL...")
comments_llm = load_and_parse_comments_by_platform(COMMENTS_LLM_PATH, PLATFORMS_TO_INCLUDE)

# Merge features with text
comment_features = comment_features.merge(
    comments_llm[["comment_id", "text"]],
    on="comment_id",
    how="left"
)
print(f"✅ Merged: {len(comment_features):,} rows with text")


# ── 4.3 User-level aggregation (PRE-COMPUTED FEATURES) ──────────────────────
def build_user_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate comment-level features to user level.
    Features already computed by data prep pipeline; we only aggregate.
    """
    df = df.copy()
    
    # Create user_platform_key for grouping (handles potential cross-platform IDs)
    df["user_platform_key"] = df["author_id"].astype(str) + "_" + df["platform"]
    
    grp = df.groupby("user_platform_key")
    
    # Aggregation dictionary — mostly MEANS (user behavior averages)
    agg_dict = {
        "author_id": grp["author_id"].first(),
        "platform": grp["platform"].first(),
        "total_comments": grp.size(),
        "unique_posts_commented": grp["media_id"].nunique(),
        
        # Pre-computed text features (from data prep) — take means
        "mean_text_length": grp["text_length"].mean(),
        "mean_word_count": grp["word_count"].mean(),
        "mean_emoji_count": grp["emoji_count"].mean(),
        "mean_unique_emoji_count": grp["unique_emoji_count"].mean(),
        "mean_emoji_entropy": grp["emoji_entropy"].mean(),
        "mean_emoji_variety_ratio": grp["emoji_variety_ratio"].mean(),
        "mean_emoji_per_word_ratio": grp["emoji_per_word_ratio"].mean(),
        "mean_url_count": grp["url_count"].mean(),
        "mean_mention_count": grp["mention_count"].mean(),
        "mean_hashtag_count": grp["hashtag_count"].mean(),
        "mean_exclamation_count": grp["exclamation_count"].mean(),
        "mean_question_count": grp["question_count"].mean(),
        "mean_avg_word_length": grp["avg_word_length"].mean(),
        
        # Engagement signals
        "pct_comments_with_links": (grp["has_links"].sum() / grp.size() * 100),
        "pct_comments_with_numbers": (grp["has_numbers"].sum() / grp.size() * 100),
    }
    
    # Handle reply detection if available
    if "reply_to_comment_id" in df.columns:
        agg_dict["reply_ratio"] = grp["reply_to_comment_id"].notna().mean()
        agg_dict["total_replies_made"] = grp["reply_to_comment_id"].notna().sum()
    
    feat = pd.DataFrame(agg_dict).reset_index()
    
    # Post concentration: unique posts / total comments
    feat["post_concentration_ratio"] = (
        feat["unique_posts_commented"] / feat["total_comments"].clip(lower=1)
    ).clip(upper=1.0)
    
    # Top comments per user (for LLM context)
    top_comments = (
        df.sort_values("text_length", ascending=False)
        .groupby("user_platform_key")
        .head(5)
        .groupby("user_platform_key")["text"]
        .apply(lambda texts: " ||| ".join(texts.astype(str).tolist()))
        .reset_index()
        .rename(columns={"text": "top_comments_sample"})
    )
    
    feat = feat.merge(top_comments, on="user_platform_key", how="left")
    
    return feat.drop(columns=["user_platform_key"])


user_features = build_user_feature_matrix(comment_features)

print(f"\n✅ User feature matrix built: {len(user_features):,} users.")
print(f"   Breakdown by platform:")
for platform in sorted(user_features["platform"].unique()):
    count = (user_features["platform"] == platform).sum()
    pct = (count / len(user_features) * 100)
    print(f"      {platform.upper():<12}: {count:>8,} users ({pct:>5.1f}%)")
print(f"   Features: Pre-computed by data prep pipeline, aggregated to user level.")


In [40]:
from vertexai.preview import caching
import datetime

# ────────────────────────────────
# CELL 5.1 — OPTIMIZED CAG (EXPLICIT VERTEX AI CONTEXT CACHING)
# ────────────────────────────────

def create_explicit_vertex_cache(system_instruction: str, context_text: str, model_name: str = MODEL_STAGE2_CLASSIFY):
    """
    Creates a server-side KV cache for the provided context and system instructions.
    This avoids re-tokenizing the context for every request in a batch.
    """
    print(f"⚙ Pre-computing Vertex AI Context Cache (TTL: 1h)...")

    cached_content = caching.CachedContent.create(
        model_name=model_name,
        system_instruction=system_instruction,
        contents=[context_text],
        ttl=datetime.timedelta(hours=1)
    )
    print(f"✅ Cache created: {cached_content.name}")
    return cached_content

print("✅ Vertex AI Explicit Caching logic (Optimized) ready.")

✅ Vertex AI Explicit Caching logic (Optimized) ready.


In [45]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — STAGE 1: EXPLORATORY TAXONOMY DISCOVERY (SAMPLE)
# ─────────────────────────────────────────────────────────────────────────────
# Based on the 3-step pipeline from Boughanmi et al. (2025) as referenced
# in the Show Reel LLMs & Prompting brief.
#
# Process:
#   1. Sample N users from the full user feature matrix
#   2. Send batches of user profiles to Gemini Flash
#   3. LLM identifies latent behavioral patterns / archetypes
#   4. Human-in-the-loop consolidation into a MECE taxonomy
#   5. Taxonomy saved to taxonomy.json for Stage 2
# ─────────────────────────────────────────────────────────────────────────────



In [ ]:
import json
import time
from tqdm import tqdm

STAGE1_SYSTEM_PROMPT = """You are an expert community analyst for Show Reel Media Group.
Your task is to identify distinct audience persona archetypes from user comment behavior across multiple platforms (Instagram, Facebook, TikTok).

You will receive a batch of commenter profiles. Each profile contains:
- Quantitative behavioral metrics (comment frequency, timing, engagement, platform affinity)
- The platform(s) where they are active
- A sample of their actual comments (pipe-separated)

Your goal is to identify RECURRING behavioral patterns across these users.
Patterns may be platform-specific (e.g., "TikTok-only commenters" vs "Instagram hyper-engagers") OR cross-platform universal archetypes.

For each pattern you observe, output a candidate persona with:
- A short codename (e.g., "SUPERFAN", "PLATFORM_HOPPER", "NICHE_COMMENTER")
- A 1-2 sentence behavioral description including platform affinity if notable
- Key distinguishing quantitative signals
- 2-3 representative verbatim comment fragments

Output ONLY a valid JSON array of persona objects. No preamble, no markdown fences.
Schema: [{"codename": str, "description": str, "signals": [str], "examples": [str]}]"""


def format_user_profile_for_stage1(row: pd.Series) -> str:
    """Format user profile as compact text, including platform context."""
    platform_str = row.get("platform", "unknown").upper()
    
    return (
        f"USER: {row['author_id'][:12]}... | {platform_str} | "
        f"Comments: {int(row['total_comments'])} | "
        f"Posts engaged: {int(row['unique_posts_commented'])} | "
        f"Activity span: {int(row['activity_span_days'])} days | "
        f"Reply ratio: {row['reply_ratio']:.0%} | "
        f"Avg likes/comment: {row['mean_likes_per_comment']:.1f} | "
        f"Word count: {row['mean_word_count']:.0f} | "
        f"Emoji rate: {row['emoji_usage_rate']:.0%} | "
        f"Question rate: {row['question_rate']:.0%} | "
        f"Sample: {str(row.get('top_comments_sample', ''))[:300]}"
    )


def run_stage1_taxonomy_discovery(
    user_features_df: pd.DataFrame,
    n_sample: int = SAMPLE_N_USERS,
    batch_size: int = BATCH_SIZE_STAGE1,
    seed: int = SAMPLE_SEED,
) -> list[dict]:
    """
    Stage 1: Taxonomy discovery across multi-platform users.
    Samples users, balancing platforms if possible.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 1 — Exploratory Taxonomy Discovery (Multi-Platform)")
    print(f"  Sample: {n_sample} users | Batch size: {batch_size}")
    print(f"  Model: {MODEL_STAGE1_EXPLORATORY}")
    print(f"{'='*60}\n")

    # Sample users, stratifying by platform if possible
    try:
        sample_df = user_features_df.groupby("platform", group_keys=False).apply(
            lambda x: x.sample(n=min(n_sample // len(user_features_df["platform"].unique()), len(x)), 
                              random_state=seed)
        ).reset_index(drop=True)
        
        # If we don't have enough per platform, fill randomly
        if len(sample_df) < n_sample:
            remaining = user_features_df.drop(sample_df.index)
            sample_df = pd.concat([
                sample_df,
                remaining.sample(n=min(n_sample - len(sample_df), len(remaining)), random_state=seed)
            ]).reset_index(drop=True)
    except:
        # Fallback: simple random sample
        sample_df = user_features_df.sample(
            n=min(n_sample, len(user_features_df)),
            random_state=seed
        ).reset_index(drop=True)

    model = get_model(MODEL_STAGE1_EXPLORATORY)
    config = get_generation_config(MAX_OUTPUT_TOKENS_STAGE1)

    all_candidates = []
    batches = [sample_df.iloc[i:i+batch_size] for i in range(0, len(sample_df), batch_size)]

    for batch_idx, batch in enumerate(tqdm(batches, desc="Stage 1 batches")):
        profiles_text = "\n\n".join(
            format_user_profile_for_stage1(row) for _, row in batch.iterrows()
        )

        prompt = f"""{STAGE1_SYSTEM_PROMPT}

--- USER PROFILES BATCH {batch_idx + 1} of {len(batches)} ---

{profiles_text}

Identify all distinct behavioral archetypes present in this batch (platform-aware).
Output ONLY the JSON array."""

        try:
            response = model.generate_content(contents=prompt, generation_config=config)
            raw = response.text.strip()
            # Clean markdown code blocks if present
            raw = re.sub(r"^```json\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            candidates = json.loads(raw)
            if isinstance(candidates, list):
                all_candidates.extend(candidates)
            print(f"   Batch {batch_idx+1}: {len(candidates)} candidates found.")
        except Exception as e:
            print(f"   ⚠️  Batch {batch_idx+1} error: {e}")

        time.sleep(1.5)

    print(f"\n✅ Stage 1 complete. Total raw candidates: {len(all_candidates)}")
    return all_candidates


def save_taxonomy_for_review(candidates: list[dict], output_path: str = TAXONOMY_JSON_PATH):
    """Save taxonomy candidates for human review."""
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    output = {
        "status": "PENDING_HUMAN_REVIEW",
        "instructions": (
            "Review and consolidate the multi-platform persona candidates below into a MECE taxonomy. "
            "Note: These personas may span multiple platforms or be platform-specific. "
            "Each final persona must have a unique 'codename'. "
            "Change status to 'APPROVED' before running Stage 2."
        ),
        "raw_candidates": candidates,
        "final_taxonomy": []
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"✅ Taxonomy candidates saved to: {output_path}")
    print("   ⚠️  HUMAN ACTION REQUIRED: Review, consolidate, and set status='APPROVED'.")


print("✅ Multi-platform Stage 1 functions ready.")


In [27]:
# prompt: show the dataset that has the output of the llm from cell above

import pandas as pd
if PIPELINE_MODE in ["SAMPLE", "ALL"]:
    # Check if the taxonomy file exists and is approved
    if os.path.exists(TAXONOMY_JSON_PATH):
        with open(TAXONOMY_JSON_PATH, "r", encoding="utf-8") as f:
            taxonomy_data = json.load(f)
            if taxonomy_data.get("status") == "APPROVED" and taxonomy_data.get("final_taxonomy"):
                print("✅ Taxonomy is approved and loaded.")
                # The final_taxonomy is what we need to display
                final_taxonomy_df = pd.DataFrame(taxonomy_data["final_taxonomy"])
                print("\nDisplaying the approved final taxonomy:")
                display(final_taxonomy_df)
            elif taxonomy_data.get("status") == "PENDING_HUMAN_REVIEW":
                print("⚠️ Taxonomy is pending human review. Please review and approve the taxonomy.json file.")
                print("   Displaying raw candidates for your convenience:")
                raw_candidates_df = pd.DataFrame(taxonomy_data.get("raw_candidates", []))
                display(raw_candidates_df)
            else:
                print("⚠️ Taxonomy file exists but is not approved or is empty. Running Stage 1 to generate candidates.")
                # If not approved, we might want to re-run Stage 1 or prompt the user to review
                # For this task, we'll assume if it's not approved, we show raw candidates if available
                raw_candidates_df = pd.DataFrame(taxonomy_data.get("raw_candidates", []))
                if not raw_candidates_df.empty:
                    print("   Displaying raw candidates from previous Stage 1 run:")
                    display(raw_candidates_df)
                else:
                    print("   No raw candidates found. Please run Stage 1 first.")
    else:
        print("⚠️ Taxonomy file not found. Please run Stage 1 to generate taxonomy candidates.")
else:
    print("Pipeline mode is not set to 'SAMPLE' or 'ALL'. Stage 1 (taxonomy discovery) is skipped.")


⚠️ Taxonomy file not found. Please run Stage 1 to generate taxonomy candidates.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — STAGE 2: DETERMINISTIC MULTI-PLATFORM CLASSIFICATION
# ─────────────────────────────────────────────────────────────────────────────
# Applies the approved taxonomy to classify ALL users across all platforms.

def build_stage2_system_prompt(taxonomy: list[dict]) -> str:
    """Build classification prompt aware of platform-specific behavior."""
    taxonomy_text = ""
    for persona in taxonomy:
        taxonomy_text += f"""
PERSONA: {persona['codename']} — {persona.get('label', 'N/A')}
  Description: {persona['description']}
  Signals: {'; '.join(persona.get('quantitative_signals', []))}
  Examples: {' | '.join(persona.get('example_comments', []))}
"""

    return f"""You are a deterministic community analyst for Show Reel Media Group.
Your task is to classify commenters into exactly ONE persona from the approved taxonomy.
Users may be active on Instagram, Facebook, TikTok, or combinations thereof.

=== APPROVED PERSONA TAXONOMY ===
{taxonomy_text}
=================================

CLASSIFICATION RULES:
1. Each user MUST be assigned to exactly ONE persona — the best match.
2. Output a confidence score (0.0–1.0) based on how well the signals align.
3. Cite specific evidence from their comments or metrics as justification.
4. Consider platform-specific behavior (TikTok comments differ from Instagram).
5. If insufficient data, assign the most probable persona with confidence ≤ 0.4.
6. Output ONLY a valid JSON array. No preamble, no markdown fences.

JSON per user:
{{
  "author_id": str,
  "platform": str,
  "persona_codename": str,    // one of the taxonomy codenames
  "confidence": float,        // 0.0 to 1.0
  "justification": str        // cite evidence
}}"""


def format_user_profile_for_stage2(row: pd.Series) -> dict:
    """Format user profile for Stage 2, including platform."""
    return {
        "author_id": str(row["author_id"]),
        "platform": str(row.get("platform", "unknown")),
        "total_comments": int(row.get("total_comments", 0)),
        "unique_posts": int(row.get("unique_posts_commented", 0)),
        "activity_span_days": int(row.get("activity_span_days", 0)),
        "reply_ratio": round(float(row.get("reply_ratio", 0)), 2),
        "mean_likes_per_comment": round(float(row.get("mean_likes_per_comment", 0)), 2),
        "mean_word_count": round(float(row.get("mean_word_count", 0)), 1),
        "emoji_usage_rate": round(float(row.get("emoji_usage_rate", 0)), 2),
        "question_rate": round(float(row.get("question_rate", 0)), 2),
        "exclamation_rate": round(float(row.get("exclamation_rate", 0)), 2),
        "mean_mention_count": round(float(row.get("mean_mention_count", 0)), 2),
        "post_concentration_ratio": round(float(row.get("post_concentration_ratio", 0)), 2),
        "sample_comments": str(row.get("top_comments_sample", ""))[:400],
    }


def run_stage2_classification(
    user_features_df: pd.DataFrame,
    taxonomy: list[dict],
    batch_size: int = BATCH_SIZE_STAGE2,
    output_path: str = RESULTS_PATH,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Stage 2: Classify all users deterministically (multi-platform).
    Supports resumption from checkpoint.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 2 — Deterministic Classification (Multi-Platform)")
    print(f"  Total users: {len(user_features_df):,} | Batch size: {batch_size}")
    print(f"  Model: {MODEL_STAGE2_CLASSIFY} | Temp: {TEMPERATURE}")
    print(f"{'='*60}\n")

    # Resume from checkpoint
    checkpoint_path = output_path.replace(".parquet", "_checkpoint.parquet")
    if resume and os.path.exists(checkpoint_path):
        done_df = pd.read_parquet(checkpoint_path)
        done_ids = set(done_df["author_id"].astype(str).tolist())
        todo_df = user_features_df[~user_features_df["author_id"].astype(str).isin(done_ids)]
        results = done_df.to_dict("records")
        print(f"   ↺  Resuming: {len(done_ids):,} done, {len(todo_df):,} remaining.")
    else:
        todo_df = user_features_df.copy()
        results = []

    system_prompt = build_stage2_system_prompt(taxonomy)
    model = get_model(MODEL_STAGE2_CLASSIFY)
    config = get_generation_config(MAX_OUTPUT_TOKENS_STAGE2)

    batches = [todo_df.iloc[i:i+batch_size] for i in range(0, len(todo_df), batch_size)]
    error_count = 0

    for batch_idx, batch in enumerate(tqdm(batches, desc="Stage 2 classification")):
        user_profiles = [format_user_profile_for_stage2(row) for _, row in batch.iterrows()]
        user_batch_text = json.dumps(user_profiles, ensure_ascii=False, indent=2)

        full_prompt = f"""{system_prompt}

=== USERS TO CLASSIFY (Batch {batch_idx + 1} of {len(batches)}) ===
{user_batch_text}

Classify each user. Output ONLY the JSON array."""

        try:
            response = model.generate_content(contents=full_prompt, generation_config=config)
            raw = response.text.strip()
            raw = re.sub(r"^```json\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            classified = json.loads(raw)

            if isinstance(classified, list):
                results.extend(classified)

            if (batch_idx + 1) % 10 == 0:
                pd.DataFrame(results).to_parquet(checkpoint_path, index=False)
                print(f"   💾 Checkpoint saved at batch {batch_idx + 1}.")

        except Exception as e:
            error_count += 1
            print(f"   ⚠️  Batch {batch_idx+1} error: {e}")
            for profile in user_profiles:
                results.append({
                    "author_id": profile["author_id"],
                    "platform": profile["platform"],
                    "persona_codename": "CLASSIFICATION_ERROR",
                    "confidence": 0.0,
                    "justification": f"API error: {str(e)[:80]}"
                })

        time.sleep(1.0)

    # Assemble final results
    results_df = pd.DataFrame(results)
    final_df = user_features_df.merge(
        results_df,
        on=["author_id", "platform"],
        how="left"
    )

    final_df.to_parquet(output_path, index=False)
    print(f"\n✅ Stage 2 complete.")
    print(f"   Users classified: {len(final_df):,}")
    print(f"   Errors: {error_count} batches")
    print(f"   Output: {output_path}")

    # Distribution by platform + persona
    print(f"\n   📊 Distribution by Platform & Persona:")
    for platform in sorted(final_df["platform"].unique()):
        platform_df = final_df[final_df["platform"] == platform]
        print(f"\n   {platform.upper()}:")
        dist = platform_df["persona_codename"].value_counts()
        for persona, count in dist.items():
            pct = (count / len(platform_df) * 100)
            print(f"      {persona:<30} {count:>6,} ({pct:>5.1f}%)")

    return final_df


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — PIPELINE ORCHESTRATOR (Multi-Platform Entry Point)
# ─────────────────────────────────────────────────────────────────────────────
# Single entry point for cross-platform persona identification.
# Set PIPELINE_MODE + PLATFORMS_TO_INCLUDE at the top to customize behavior.

def load_approved_taxonomy(path: str = TAXONOMY_JSON_PATH) -> list[dict]:
    """Load the human-approved MECE taxonomy from disk."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if data.get("status") != "APPROVED":
        raise ValueError(
            f"Taxonomy at '{path}' has status: {data.get('status')}. "
            "Complete human review and set status='APPROVED' before Stage 2."
        )

    taxonomy = data.get("final_taxonomy", [])
    if not taxonomy:
        raise ValueError("final_taxonomy is empty. Fill in personas before Stage 2.")

    print(f"✅ Approved taxonomy loaded: {len(taxonomy)} personas.")
    return taxonomy


def run_pipeline(mode: str = PIPELINE_MODE):
    """
    Multi-platform persona pipeline orchestrator.
    mode = "SAMPLE"  → Stage 0 + Stage 1 (taxonomy discovery across platforms)
    mode = "FULL"    → Stage 0 + Stage 2 (classification across platforms)
    mode = "ALL"     → Stage 0 + Stage 1 + Stage 2
    """
    print(f"\n{'#'*70}")
    print(f"  MULTI-PLATFORM PERSONA IDENTIFICATION PIPELINE")
    print(f"  Mode: {mode} | Platforms: {', '.join(PLATFORMS_TO_INCLUDE)}")
    print(f"{'#'*70}\n")

    # Stage 0 runs first (comment loading and feature engineering)
    # Done implicitly in earlier cells: comments_df → user_features

    if mode in ("SAMPLE", "ALL"):
        print("⏳ STAGE 1: Exploratory Taxonomy Discovery...")
        candidates = run_stage1_taxonomy_discovery(
            user_features_df=user_features,
            n_sample=SAMPLE_N_USERS,
            batch_size=BATCH_SIZE_STAGE1,
            seed=SAMPLE_SEED,
        )
        save_taxonomy_for_review(candidates, TAXONOMY_JSON_PATH)

        if mode == "SAMPLE":
            print("\n" + "="*70)
            print("⏸  STAGE 1 COMPLETE — Pipeline paused for human review.")
            print("="*70)
            print(f"   📋 Review and consolidate personas at: {TAXONOMY_JSON_PATH}")
            print(f"   ✅ After approval, set PIPELINE_MODE='FULL' or 'ALL' and re-run.")
            return None

    if mode in ("FULL", "ALL"):
        print("\n⏳ STAGE 2: Deterministic Classification...")
        taxonomy = load_approved_taxonomy(TAXONOMY_JSON_PATH)
        final_df = run_stage2_classification(
            user_features_df=user_features,
            taxonomy=taxonomy,
            batch_size=BATCH_SIZE_STAGE2,
            output_path=RESULTS_PATH,
            resume=True,
        )
        
        print("\n" + "="*70)
        print("✅ PIPELINE COMPLETE")
        print("="*70)
        print(f"   📊 Results saved to: {RESULTS_PATH}")
        print(f"   👥 Total users profiled: {len(final_df):,}")
        return final_df

    return None


# ── EXECUTE ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    result = run_pipeline(mode=PIPELINE_MODE)


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — OPTIONAL: UPLOAD OUTPUTS TO GCS
# ─────────────────────────────────────────────────────────────────────────────

def upload_outputs_to_gcs(local_dir: str = "outputs/", gcs_path: str = GCS_BUCKET):
    """Upload all pipeline outputs to GCS for persistence across runtimes."""
    from google.cloud import storage

    client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = gcs_path.replace("gs://", "").split("/")[0]
    prefix      = "/".join(gcs_path.replace("gs://", "").split("/")[1:])
    bucket      = client.bucket(bucket_name)

    for fname in os.listdir(local_dir):
        local_path = os.path.join(local_dir, fname)
        blob_path  = f"{prefix}/{fname}" if prefix else fname
        blob       = bucket.blob(blob_path)
        blob.upload_from_filename(local_path)
        print(f"   ☁️  Uploaded: {local_path} → gs://{bucket_name}/{blob_path}")

# Uncomment to upload after pipeline completes:
# upload_outputs_to_gcs()



In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — VALIDATION & QUICK QA
# ─────────────────────────────────────────────────────────────────────────────

def validate_classification_output(results_path: str = RESULTS_PATH):
    """
    Quick QA checks on the classification output:
    - Coverage rate (% users successfully classified)
    - Confidence distribution
    - MECE check (each user has exactly one persona)
    - Low-confidence flagging (< 0.4)
    """
    df = pd.read_parquet(results_path)

    total        = len(df)
    classified   = df["persona_codename"].notna() & \
                   (df["persona_codename"] != "CLASSIFICATION_ERROR")
    coverage_pct = classified.sum() / total * 100

    print(f"\n📋 CLASSIFICATION QA REPORT")
    print(f"   Total users        : {total:,}")
    print(f"   Successfully classified: {classified.sum():,} ({coverage_pct:.1f}%)")
    print(f"   Errors             : {(~classified).sum():,}")

    sub = df[classified].copy()
    sub["confidence"] = pd.to_numeric(sub["confidence"], errors="coerce")

    print(f"\n   Confidence Distribution:")
    print(f"     Mean   : {sub['confidence'].mean():.3f}")
    print(f"     Median : {sub['confidence'].median():.3f}")
    print(f"     < 0.40 : {(sub['confidence'] < 0.4).sum():,} users (low confidence, flag for review)")

    # MECE check: no user appears twice
    dupes = df["from_id"].duplicated().sum()
    print(f"\n   MECE Check: {dupes} duplicate user assignments "
          f"({'⚠️  FAIL' if dupes > 0 else '✅ PASS'})")

    return df

# Uncomment after Stage 2 completes:
# qa_df = validate_classification_output()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — EXAMPLE: PROCESS & EXPORT SAMPLE COMMENTS TO JSONL
# ─────────────────────────────────────────────────────────────────────────────
# Demonstrates the comment preprocessing pipeline with a small sample.
# Set max_records to test before running on the full dataset.

# Test with a small sample first (e.g., 100 comments)
print("🧪 Testing comment preprocessing on a sample...\n")

# Apply preprocessing to a small sample
sample_size = 100
sample_comments = ig_comments.head(sample_size)
sample_preprocessed = sample_comments["text"].apply(advanced_text_preprocessing)

# Display stats
print(f"Sample Stats ({sample_size} comments):")
print(f"  Avg emoji count:    {sample_preprocessed.apply(lambda x: x['emoji_count']).mean():.2f}")
print(f"  Avg mention count:  {sample_preprocessed.apply(lambda x: x['mention_count']).mean():.2f}")
print(f"  Avg hashtag count:  {sample_preprocessed.apply(lambda x: x['hashtag_count']).mean():.2f}")
print(f"  Language diversity: {len(sample_preprocessed.apply(lambda x: x['language']).unique())} unique languages")
print(f"  Mixed-script text:  {sample_preprocessed.apply(lambda x: x['has_multiple_languages']).sum()} comments")

# Show 3 examples of processed comments
print(f"\n📋 Sample processed comments:\n")
for i in range(min(3, len(sample_preprocessed))):
    proc = sample_preprocessed.iloc[i]
    print(f"  Example {i+1}:")
    print(f"    Original:  {proc['text_original'][:80]}")
    print(f"    Cleaned:   {proc['text_cleaned'][:80]}")
    print(f"    Language:  {proc['language']} | Emoji: {proc['emoji_count']} | Mentions: {proc['mention_count']}")
    print()

# ── Full export (uncomment to run on entire dataset) ──────────────────────────
# To export ALL comments (may take a few minutes):
#
# export_comments_to_jsonl(
#     comments_df=ig_comments,
#     output_gcs_path=f"gs://{GCS_BUCKET}/ig_comments_for_llm.jsonl",
#     include_features=True,
#     max_records=None
# )

# ── Or test with a smaller subset for validation ──────────────────────────────
# To test with 5000 comments first:
#
# export_comments_to_jsonl(
#     comments_df=ig_comments,
#     output_gcs_path=f"gs://{GCS_BUCKET}/ig_comments_for_llm_sample_5k.jsonl",
#     include_features=True,
#     max_records=5000
# )

print("\n✅ Preprocessing pipeline tested. Ready to export full dataset when needed.")